# Phase 1: 数据集整理 (DataSet exposing)

## 概述 Overview

本Notebook文件基于Phase1脚本修改而来，目标为按照一定比例，融合不同专利的数据集，形成测试集和验证集，并且将测试集合和验证集合命名为feature-engineering_{train/test}.csv和npz文件
筛选操作也是同样做处理

另外，为了配合更多分子表征的操作，这里新增了ChemBerta分子表征的内容，ChemBerta分子表征默认数值为384，这个暂时还不知道怎么处理

**更新内容**

- 该脚本主要是获取公司需要的数据
- 在Phase1阶段就筛选对应的数据（is_monomer=True， 对应半衰期不为-1）
- **特殊情况需要注意：由于id失效导致Phase3中的筛选逻辑基本失效**
- **csv数据和npz特征数据只能是按顺序对应的内容**

**主要步骤：**
* 4268数据集合完全划分到训练集中
* 剩下的所有数据7为训练集，3为测试集合
* 经过数据集划分验证，得知当前的划分方式基本合理
* 新增了ChemBerta分子表征内容
* **重要更新：只获取monomer分子的内容**

**生成结果**
* csv文件（没有processed后缀，存放在data/general/csv文件夹下方）
* npz文件（没有processed后缀，存放在data/general/feature下方）

**输入**: `data/raw/*.csv` - 原始数据文件  
**输出**:  输出的每种文件包含Train/Test_sif/sgf两种类型，所以csv和npz各为4种类
- `data/feature-engineering/csv/*.csv` - 添加分子特征后的CSV
- `data/feature-engineering/feature/*.npz` - RDKit特征矩阵   


---

## 1. 环境检查与导入 Environment Setup

In [1]:
# 环境检查
import sys
from pathlib import Path

# 添加项目根目录到路径
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# 核心库导入
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# 项目模块导入
from feature_extraction import PeptideFeaturizer
from feature_extraction.utils import (
    get_csv_files, load_csv_safely, extract_molecular_features,
    convert_label_to_minutes, save_features_to_npz
)

# 设置显示选项
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

print("✓ 所有库已成功导入")
print(f"✓ 项目根目录: {project_root}")




✓ 所有库已成功导入
✓ 项目根目录: d:\RA\feature_extraction


## 2. 参数配置区 Configuration

**⚙️ 根据您的需求修改以下参数**

In [2]:
# ============== 参数配置区 ==============
# 用户可根据需要修改以下参数

CONFIG = {
    # 读入数据路径
    'raw_dir': project_root / 'data' / 'raw',
    # csv输出路径
    'processed_dir': project_root / 'data' / 'general' / 'csv',
    'features_dir': project_root / 'data' / 'general' / 'features',
    #'figures_dir': project_root / 'data' / 'general'/ 'figures' / 'phase1',

    # 备注信息：（用于挑选对应的数据和特征）
    'message':"Morgan(1024)_Avalon(512)",#Morgan(1024)_Avalon(512)_ChemBERTa(384)
    
    # 特征提取参数
    'morgan_bits': 1024, #1024,     # Morgan指纹位数（这里临时修改为0，关闭特征提取）
    'avalon_bits': 512,      # Avalon指纹位数512
    'chemberta_bits': 0,   # 根据目前所知，这个只有0和384，剩下改了都没用
    # （放在外面配置了）'use_avalon': True ,      # 是否使用Avalon指纹（需RDKit支持） 

    # 可视化参数
    'dpi': 300,              # 图像分辨率
    'format': 'png',         # 图像格式 (png/pdf/svg)
    'display_plots': True,   # 是否在notebook中显示关键图表
    'max_display_plots': 3,  # 最多显示几个图表
    'split_random':90
}

# 追加配置
CONFIG['use_avalon'] = True if CONFIG['avalon_bits'] > 0 else False  # 是否使用Avalon指纹（需RDKit支持）
CONFIG['use_chemberta'] = True if CONFIG['chemberta_bits'] > 0 else False  # 是否使用Avalon指纹（需RDKit支持）

# 创建输出目录
CONFIG['processed_dir'].mkdir(parents=True, exist_ok=True)
CONFIG['features_dir'].mkdir(parents=True, exist_ok=True)
# CONFIG['figures_dir'].mkdir(parents=True, exist_ok=True)

print("配置参数:")
for key, value in CONFIG.items():
    if isinstance(value, Path):
        print(f"  {key}: {value.relative_to(project_root) if value.is_relative_to(project_root) else value}")
    else:
        print(f"  {key}: {value}")

配置参数:
  raw_dir: data\raw
  processed_dir: data\general\csv
  features_dir: data\general\features
  message: Morgan(1024)_Avalon(512)
  morgan_bits: 1024
  avalon_bits: 512
  chemberta_bits: 0
  dpi: 300
  format: png
  display_plots: True
  max_display_plots: 3
  split_random: 90
  use_avalon: True
  use_chemberta: False


## 3. 步骤 1: 添加分子特征 Add Molecular Features

为每个SMILES分子添加结构特征并转换标签格式。

**新增列**:
- `is_dimer`: 是否为二聚体 (bool)
- `is_cyclic`: 是否含环状结构 (bool)
- `has_disulfide_bond`: 是否含二硫键 (bool)
- `SIF_minutes`: SIF半衰期（分钟）
- `SGF_minutes`: SGF半衰期（分钟）

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os
from tqdm import tqdm

def process_and_merge_datasets(file_paths, full_train_names, train_ratio=0.7, message:str=""):
    """
    file_paths: list, 所有数据集的文件路径列表
    full_train_names: list, 需要全部作为训练集的文件名关键词（如 US9624268）
    train_ratio: float, 其他数据集的训练集比例
    """

    # =========================
    # 为两个任务分别准备容器
    # =========================
    train_list_sif, test_list_sif = [], []
    train_list_sgf, test_list_sgf = [], []

    for path in file_paths:
        file_name = path
        csv_path = CONFIG['raw_dir'] / path
        print(f"正在处理: {file_name}")
        df = pd.read_csv(csv_path)

        # ==================================================
        # 【清理废数据】全局生效
        # 1. SMILES 不能为空
        # ==================================================
        df = df[df["SMILES"].notna()].copy()

        # ------------------ 原有预处理逻辑开始 ------------------
        # 提取分子特征
        feature_records = []
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"处理 {file_name}", leave=False):
            features = extract_molecular_features(row["SMILES"])
            feature_records.append(features)

        df = pd.concat([df, pd.DataFrame(feature_records)], axis=1)

        # 转换标签到分钟
        if "SIF_class" in df.columns:
            df["SIF_minutes"] = df["SIF_class"].apply(convert_label_to_minutes)
        else:
            df["SIF_minutes"] = -1

        if "SGF_class" in df.columns:
            df["SGF_minutes"] = df["SGF_class"].apply(convert_label_to_minutes)
        else:
            df["SGF_minutes"] = -1

        # ==================================================
        # 【清理废数据】
        # SIF 和 SGF 同时为 -1 的样本直接删除
        # ==================================================
        mask_both_missing = (df["SIF_minutes"] == -1) & (df["SGF_minutes"] == -1)
        df_processed = df[~mask_both_missing].copy()
        # 提前筛选：只保留 monomer
        if "is_monomer" in df_processed.columns:
            df_processed = df_processed[df_processed["is_monomer"] == True].copy()
        else:
            print(f"--- 警告: {file_name} 中不存在 is_monomer 字段")

        # 记录数据来源（用于后续分析 / 泄露检查）
        df_processed["source_name"] = file_name

        # ------------------ 原有预处理逻辑结束 ------------------

        # 是否为 4268 数据集（全量进训练集）
        is_full_train = any(name in file_name for name in full_train_names)

        # ==================================================
        # ===================== SIF 任务 ====================
        # ==================================================
        sif_df = df_processed[df_processed["SIF_minutes"] != -1].copy()
        sif_df["label"] = (sif_df["SIF_minutes"] >= 270).astype(int)

        if not sif_df.empty:
            if is_full_train:
                # 4268：SIF 有效数据全进训练集
                train_list_sif.append(sif_df)
            else:
                if sif_df["label"].nunique() < 2:
                    # 无法分层，全部进训练集
                    train_list_sif.append(sif_df)
                else:
                    train_df, test_df = train_test_split(
                        sif_df,
                        train_size=train_ratio,
                        stratify=sif_df["label"],
                        random_state=CONFIG["split_random"]
                    )
                    train_list_sif.append(train_df)
                    test_list_sif.append(test_df)

        # ==================================================
        # ===================== SGF 任务 ====================
        # ==================================================
        sgf_df = df_processed[df_processed["SGF_minutes"] != -1].copy()
        sgf_df["label"] = (sgf_df["SGF_minutes"] >= 250).astype(int)

        if not sgf_df.empty:
            if is_full_train:
                # 4268：SGF 有效数据全进训练集
                train_list_sgf.append(sgf_df)
            else:
                if sgf_df["label"].nunique() < 2:
                    train_list_sgf.append(sgf_df)
                else:
                    train_df, test_df = train_test_split(
                        sgf_df,
                        train_size=train_ratio,
                        stratify=sgf_df["label"],
                        random_state=CONFIG["split_random"]
                    )
                    train_list_sgf.append(train_df)
                    test_list_sgf.append(test_df)

    # ==================================================
    # 合并 & 保存
    # ==================================================
    final_train_sif = pd.concat(train_list_sif, ignore_index=True)
    final_test_sif  = pd.concat(test_list_sif, ignore_index=True) if test_list_sif else pd.DataFrame()

    final_train_sgf = pd.concat(train_list_sgf, ignore_index=True)
    final_test_sgf  = pd.concat(test_list_sgf, ignore_index=True) if test_list_sgf else pd.DataFrame()

    final_train_sif.to_csv(CONFIG["processed_dir"]/f"Train_sif_{message}.csv", index=False)
    final_test_sif.to_csv(CONFIG["processed_dir"]/f"Test_sif_{message}.csv", index=False)
    final_train_sgf.to_csv(CONFIG["processed_dir"]/f"Train_sgf_{message}.csv", index=False)
    final_test_sgf.to_csv(CONFIG["processed_dir"]/f"Test_sgf_{message}.csv", index=False)

    print("\n" + "="*30)
    print("数据处理完成")
    print(f"SIF  Train: {len(final_train_sif)} | Test: {len(final_test_sif)}")
    print(f"SGF  Train: {len(final_train_sgf)} | Test: {len(final_test_sgf)}")
    print("="*30)


csv_files = list(CONFIG['raw_dir'].glob('*.csv'))
csv_files_names = [file.name for file in csv_files]
print(f"找到 {len(csv_files)} 个CSV文件\n")
special_datasets = ['US9624268.csv']
process_and_merge_datasets(csv_files_names, special_datasets, message=CONFIG['message'])


找到 5 个CSV文件

正在处理: sif_sgf_second.csv


正在处理: US20140294902A1.csv


正在处理: US9624268.csv


正在处理: US9809623B2.csv


正在处理: WO2017011820A2.csv



数据处理完成
SIF  Train: 459 | Test: 56
SGF  Train: 370 | Test: 32


## 4. 步骤 2: 提取RDKit特征 Extract RDKit Features

从处理后的CSV中提取分子特征向量，保存为NPZ格式。

**特征类型**:
- QED属性 (8维)
- 物理化学描述符 (11维)
- Gasteiger电荷统计 (5维)
- Morgan指纹 (1024维)
- Avalon指纹 (512维, 可选)

In [4]:
from pathlib import Path
def extract_rdkit_features(csv_path: Path, output_dir: Path, featurizer):
    """
    从CSV提取RDKit特征并保存为NPZ
    
    Args:
        csv_path: 输入CSV文件路径
        output_dir: 输出目录
        featurizer: PeptideFeaturizer实例
    
    Returns:
        dict: 统计信息
    """
    # 加载CSV
    df, _ = load_csv_safely(csv_path, required_columns=["id", "SMILES", "SIF_minutes", "SGF_minutes"])
    if df is None:
        return {"error": "Failed to load CSV"}
    
    X = []
    y_sif = []
    y_sgf = []
    ids = []
    valid_count = 0
    
    # 提取特征
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"提取特征 {csv_path.name}", leave=False):
        smiles = str(row["SMILES"])
        features, success = featurizer.featurize(smiles)
        
        if success and features is not None:
            X.append(features)
            y_sif.append(int(row["SIF_minutes"]) if not pd.isna(row["SIF_minutes"]) else -1)
            y_sgf.append(int(row["SGF_minutes"]) if not pd.isna(row["SGF_minutes"]) else -1)
            ids.append(str(row["id"]))
            valid_count += 1
    
    # 转换为NumPy数组
    X = np.array(X, dtype=np.float32)
    y_sif = np.array(y_sif, dtype=np.int32)
    y_sgf = np.array(y_sgf, dtype=np.int32)
    ids = np.array(ids, dtype=object)
    feature_names = featurizer.get_feature_names()
    
    # 保存NPZ
    output_path = output_dir / csv_path.name.replace('.csv', '.npz')
    np.savez_compressed(
        output_path,
        X=X,
        y_sif=y_sif,
        y_sgf=y_sgf,
        ids=ids,
        feature_names=feature_names,
    )
    
    return {
        "file": csv_path.name,
        "total_samples": len(df),
        "valid_samples": valid_count,
        "feature_dim": X.shape[1],
        "output_path": output_path,
    }

# 初始化特征提取器
featurizer = PeptideFeaturizer(
    morgan_bits=CONFIG['morgan_bits'],
    avalon_bits=CONFIG['avalon_bits'],
    chemberta_bits=CONFIG['chemberta_bits'],
    use_avalon=CONFIG['use_avalon'],
    use_chemberta=CONFIG['use_chemberta']
)

print(f"特征提取器配置:")
print(f"  Morgan指纹: {CONFIG['morgan_bits']} bits")
print(f"  Avalon指纹: {CONFIG['avalon_bits']} bits (启用: {CONFIG['use_avalon']})")
print(f"  ChemBerta指纹: {CONFIG['chemberta_bits']} bits (启用: {CONFIG['use_chemberta']})")
print(f"  预计总特征维度: {featurizer.n_features}\n")

# 执行：批量提取测试集特征
processed_csvs = list(CONFIG['processed_dir'].glob(f"Test_*_{CONFIG['message']}.csv"))
print(f"找到 {len(processed_csvs)} 个处理后的CSV文件\n")

feature_stats = []
for csv_file in processed_csvs:
    stats = extract_rdkit_features(csv_file, CONFIG['features_dir'], featurizer)
    if "error" not in stats:
        feature_stats.append(stats)
        print(f"✓ {stats['file']}: {stats['valid_samples']} samples, {stats['feature_dim']} features")

# 执行：批量提取训练集特征
processed_csvs = list(CONFIG['processed_dir'].glob(f"Train_*_{CONFIG['message']}.csv"))
print(f"找到 {len(processed_csvs)} 个处理后的CSV文件\n")

feature_stats = []
for csv_file in processed_csvs:
    stats = extract_rdkit_features(csv_file, CONFIG['features_dir'], featurizer)
    if "error" not in stats:
        feature_stats.append(stats)
        print(f"✓ {stats['file']}: {stats['valid_samples']} samples, {stats['feature_dim']} features")

# 汇总
feat_summary_df = pd.DataFrame(feature_stats)
print(f"\n{'='*60}")
print("特征提取总结:")
print(f"  总样本数: {feat_summary_df['valid_samples'].sum()}")
print(f"  特征维度: {feat_summary_df['feature_dim'].iloc[0]}")
print(f"{'='*60}\n")

display(feat_summary_df[['file', 'total_samples', 'valid_samples', 'feature_dim']])

特征提取器配置:
  Morgan指纹: 1024 bits
  Avalon指纹: 512 bits (启用: True)
  ChemBerta指纹: 0 bits (启用: False)
  预计总特征维度: 1560

找到 2 个处理后的CSV文件



提取特征 Test_sgf_Morgan(1024)_Avalon(512).csv:   0%|          | 0/32 [00:00<?, ?it/s][16:00:00] DEPRECATION WARNING: please use MorganGenerator
[16:00:00] DEPRECATION WARNING: please use MorganGenerator
[16:00:00] DEPRECATION WARNING: please use MorganGenerator
提取特征 Test_sgf_Morgan(1024)_Avalon(512).csv:   9%|▉         | 3/32 [00:00<00:01, 23.14it/s][16:00:00] DEPRECATION WARNING: please use MorganGenerator
[16:00:00] DEPRECATION WARNING: please use MorganGenerator
[16:00:00] DEPRECATION WARNING: please use MorganGenerator
提取特征 Test_sgf_Morgan(1024)_Avalon(512).csv:  19%|█▉        | 6/32 [00:00<00:02,  9.73it/s][16:00:00] DEPRECATION WARNING: please use MorganGenerator
[16:00:00] DEPRECATION WARNING: please use MorganGenerator
[16:00:00] DEPRECATION WARNING: please use MorganGenerator
提取特征 Test_sgf_Morgan(1024)_Avalon(512).csv:  28%|██▊       | 9/32 [00:00<00:02, 11.42it/s][16:00:00] DEPRECATION WARNING: please use MorganGenerator
[16:00:00] DEPRECATION WARNING: please use MorganGenerator

✓ Test_sgf_Morgan(1024)_Avalon(512).csv: 32 samples, 1560 features


提取特征 Test_sif_Morgan(1024)_Avalon(512).csv:   0%|          | 0/56 [00:00<?, ?it/s][16:00:02] DEPRECATION WARNING: please use MorganGenerator
[16:00:02] DEPRECATION WARNING: please use MorganGenerator
[16:00:02] DEPRECATION WARNING: please use MorganGenerator
[16:00:02] DEPRECATION WARNING: please use MorganGenerator
提取特征 Test_sif_Morgan(1024)_Avalon(512).csv:   7%|▋         | 4/56 [00:00<00:01, 33.12it/s][16:00:02] DEPRECATION WARNING: please use MorganGenerator
[16:00:02] DEPRECATION WARNING: please use MorganGenerator
[16:00:02] DEPRECATION WARNING: please use MorganGenerator
[16:00:02] DEPRECATION WARNING: please use MorganGenerator
[16:00:02] DEPRECATION WARNING: please use MorganGenerator
[16:00:02] DEPRECATION WARNING: please use MorganGenerator
提取特征 Test_sif_Morgan(1024)_Avalon(512).csv:  18%|█▊        | 10/56 [00:00<00:01, 44.42it/s][16:00:02] DEPRECATION WARNING: please use MorganGenerator
[16:00:02] DEPRECATION WARNING: please use MorganGenerator
[16:00:02] DEPRECATION WARNIN

✓ Test_sif_Morgan(1024)_Avalon(512).csv: 56 samples, 1560 features
找到 2 个处理后的CSV文件



提取特征 Train_sgf_Morgan(1024)_Avalon(512).csv:   0%|          | 0/370 [00:00<?, ?it/s][16:00:04] DEPRECATION WARNING: please use MorganGenerator
[16:00:04] DEPRECATION WARNING: please use MorganGenerator
提取特征 Train_sgf_Morgan(1024)_Avalon(512).csv:   1%|          | 2/370 [00:00<00:19, 19.07it/s][16:00:04] DEPRECATION WARNING: please use MorganGenerator
[16:00:04] DEPRECATION WARNING: please use MorganGenerator
提取特征 Train_sgf_Morgan(1024)_Avalon(512).csv:   1%|          | 4/370 [00:00<00:22, 16.39it/s][16:00:04] DEPRECATION WARNING: please use MorganGenerator
[16:00:04] DEPRECATION WARNING: please use MorganGenerator
提取特征 Train_sgf_Morgan(1024)_Avalon(512).csv:   2%|▏         | 6/370 [00:00<00:22, 16.31it/s][16:00:04] DEPRECATION WARNING: please use MorganGenerator
[16:00:04] DEPRECATION WARNING: please use MorganGenerator
提取特征 Train_sgf_Morgan(1024)_Avalon(512).csv:   2%|▏         | 8/370 [00:00<00:21, 16.47it/s][16:00:04] DEPRECATION WARNING: please use MorganGenerator
[16:00:04] DEPREC

✓ Train_sgf_Morgan(1024)_Avalon(512).csv: 370 samples, 1560 features


提取特征 Train_sif_Morgan(1024)_Avalon(512).csv:   0%|          | 0/459 [00:00<?, ?it/s][16:00:22] DEPRECATION WARNING: please use MorganGenerator
[16:00:22] DEPRECATION WARNING: please use MorganGenerator
提取特征 Train_sif_Morgan(1024)_Avalon(512).csv:   0%|          | 2/459 [00:00<00:26, 17.38it/s][16:00:22] DEPRECATION WARNING: please use MorganGenerator
[16:00:22] DEPRECATION WARNING: please use MorganGenerator
[16:00:22] DEPRECATION WARNING: please use MorganGenerator
提取特征 Train_sif_Morgan(1024)_Avalon(512).csv:   1%|          | 5/459 [00:00<00:25, 18.08it/s][16:00:22] DEPRECATION WARNING: please use MorganGenerator
[16:00:22] DEPRECATION WARNING: please use MorganGenerator
提取特征 Train_sif_Morgan(1024)_Avalon(512).csv:   2%|▏         | 7/459 [00:00<00:26, 17.04it/s][16:00:22] DEPRECATION WARNING: please use MorganGenerator
[16:00:22] DEPRECATION WARNING: please use MorganGenerator
提取特征 Train_sif_Morgan(1024)_Avalon(512).csv:   2%|▏         | 9/459 [00:00<00:27, 16.63it/s][16:00:22] DEPREC

✓ Train_sif_Morgan(1024)_Avalon(512).csv: 459 samples, 1560 features

特征提取总结:
  总样本数: 829
  特征维度: 1560



,file,total_samples,valid_samples,feature_dim
0,Train_sgf_Morgan(1024)_Avalon(512).csv,370,370,1560
1,Train_sif_Morgan(1024)_Avalon(512).csv,459,459,1560


## 5. 结果总结

Phase 1 数据转化已完成！

In [5]:
print("="*70)
print("Phase 1: 数据转化 - 执行完毕")
print("="*70)

print("\n📁 目前已有的文件:")
print(f"\n  1. 处理后的CSV ({len(list(CONFIG['processed_dir'].glob('*.csv')))} 个文件):")
for f in sorted(CONFIG['processed_dir'].glob('*_processed.csv')):
    print(f"     - {f.name}")

print(f"\n  2. 特征NPZ文件 ({len(list(CONFIG['features_dir'].glob('*.npz')))} 个文件):")
for f in sorted(CONFIG['features_dir'].glob('*.npz')):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"     - {f.name} ({size_mb:.2f} MB)")



print("\n📊 处理统计:")

Phase 1: 数据转化 - 执行完毕

📁 目前已有的文件:

  1. 处理后的CSV (4 个文件):

  2. 特征NPZ文件 (4 个文件):
     - Test_sgf_Morgan(1024)_Avalon(512).npz (0.02 MB)
     - Test_sif_Morgan(1024)_Avalon(512).npz (0.03 MB)
     - Train_sgf_Morgan(1024)_Avalon(512).npz (0.13 MB)
     - Train_sif_Morgan(1024)_Avalon(512).npz (0.16 MB)

📊 处理统计:
